# Exercise 03: Slowly Changing Dimensions (SCD Type 2)

In the real world, data changes over time. A customer moves house. An employee gets promoted. A student changes their email address.

A **Slowly Changing Dimension (SCD)** is a pattern for handling these changes in a data warehouse.
There are several types — we'll implement **Type 2**, which is the most common.

---

### SCD Type 2: Keep the full history

Instead of overwriting the old row when something changes, we:
1. **Close** the old row by setting an end date and marking it inactive
2. **Insert** a new row with the updated values and a new start date

This means we can always answer: *"What did this record look like on a specific date?"*

| `student_id` | `name` | `email_address` | `dbt_valid_from` | `dbt_valid_to` | `is_current` |
|---|---|---|---|---|---|
| 63522 | Sukhil Patel | iamsohumble@email.com | 2024-01-01 | 2024-04-24 | FALSE |
| 63522 | Sukhil Patel | theverymrsveen@email.com | 2024-04-24 | NULL | TRUE |

The NULL `dbt_valid_to` means "this record is still current".

---

### The three SCD tracking columns

| Column | Type | Meaning |
|---|---|---|
| `dbt_valid_from` | `DATE` | When this version of the record became active |
| `dbt_valid_to` | `DATE` | When this version expired (NULL = still current) |
| `is_current` | `BOOLEAN` | TRUE for the active version of each record |

> These column names follow the dbt convention — the tool that popularised this pattern.

## The Data

You have two files:

- `../data/scd/students_old.csv` — the student records as they exist **today** (your initial load)
- `../data/scd/students_new.csv` — the updated records that arrive **tomorrow** (your incremental load)

Some students have changed their details between the two files. Your job is to find those changes and record the full history in a snapshot table.

**Source columns:** `student_id`, `name`, `email_address`, `phone_number`, `home_address`, `updated_at`

## Setup

In [1]:
import duckdb

con = duckdb.connect()
print('Connected! DuckDB version:', duckdb.__version__)

Connected! DuckDB version: 1.5.2


---
## Step 1: Load the Initial Student Data

Load `students_old.csv` into a table called `students`.
This represents the source data as it exists today — your initial snapshot.

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE students AS
    SELECT * FROM read_csv_auto('../data/scd/students_old.csv')
''')

con.execute('SELECT * FROM students').df()

---
## Step 2: Create `dim_students`

Create a clean dimension table from the raw `students` source.
For now, this is just a direct copy — in a real project you might rename columns or clean values.

**Columns:** `student_id`, `name`, `email_address`, `phone_number`, `home_address`, `updated_at`

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE dim_students AS
    SELECT
        -- ✏️ Select all columns from students

    FROM students
''')

In [ ]:
# Check your work — should have 6 rows, one per student
con.execute('SELECT * FROM dim_students').df()

---
## Step 3: Create `snapshot_students` — your SCD Type 2 table

Create the snapshot table from `dim_students`.
Add the three SCD tracking columns to every row:

- `dbt_valid_from` = `CURRENT_DATE` (the record is valid starting today)
- `dbt_valid_to` = `NULL` (cast to `DATE`) — no end date yet, it's current
- `is_current` = `TRUE`

**Hint:** Cast NULL to DATE like this: `NULL::DATE AS dbt_valid_to`

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE snapshot_students AS
    SELECT
        student_id,
        name,
        email_address,
        phone_number,
        home_address,
        updated_at,
        -- ✏️ Add the three SCD tracking columns below

    FROM dim_students
''')

In [ ]:
# Check your work — should have 6 rows, all with is_current = TRUE and dbt_valid_to = NULL
con.execute('SELECT * FROM snapshot_students').df()

---
## Step 4: A new day arrives — load the updated data

Load `students_new.csv` into a staging table.
This represents the latest version of student records coming from the source system.

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE students_new AS
    SELECT * FROM read_csv_auto('../data/scd/students_new.csv')
''')

con.execute('SELECT * FROM students_new').df()

---
## Step 5: Find what changed

Before updating the snapshot, identify which students have changed.

Write a query that:
- JOINs `snapshot_students` (where `is_current = TRUE`) with `students_new` on `student_id`
- Returns only rows where **any** of the tracked columns differ: `name`, `email_address`, `phone_number`, `home_address`

You should find **4 students** who have changed.

In [ ]:
# ✏️ Write a query to find students whose details have changed
# Compare snapshot_students (old/current) with students_new
con.execute('''
    SELECT
        old.student_id,
        old.name        AS old_name,        new.name        AS new_name,
        old.email_address AS old_email,     new.email_address AS new_email,
        old.phone_number  AS old_phone,     new.phone_number  AS new_phone,
        old.home_address  AS old_address,   new.home_address  AS new_address
    FROM snapshot_students old
    JOIN students_new new ON old.student_id = new.student_id
    WHERE old.is_current = TRUE
    AND (
        -- ✏️ Add your conditions to detect changes here

    )
''').df()

---
## Step 6: Apply SCD Type 2 Updates

Now that you know what changed, apply the SCD Type 2 logic in two steps:

1. **Close** the old records for changed students (set `dbt_valid_to` and `is_current = FALSE`)
2. **Insert** new, updated records for those students (with `dbt_valid_from = CURRENT_DATE`)

There's also a bonus step for any brand-new students who appear in the new file.

### Step 6a: Close the outdated records

UPDATE `snapshot_students` for the rows that have changed:
- Set `dbt_valid_to = CURRENT_DATE`
- Set `is_current = FALSE`

Target only the rows where `is_current = TRUE` and the student appears in the changed set.

In [ ]:
con.execute('''
    UPDATE snapshot_students
    SET
        dbt_valid_to = CURRENT_DATE,
        is_current   = FALSE
    WHERE is_current = TRUE
    AND student_id IN (
        -- ✏️ Subquery: find the student_ids of changed records
        -- Hint: JOIN snapshot_students with students_new and check for differences

    )
''')

print('Old records closed.')

# Check: how many records are now closed?
con.execute('''
    SELECT is_current, COUNT(*) AS num_records
    FROM snapshot_students
    GROUP BY is_current
''').df()

### Step 6b: Insert the new versions of changed records

INSERT new rows into `snapshot_students` for the students whose records you just closed.

The new rows should have:
- Values from `students_new` (the updated data)
- `dbt_valid_from = CURRENT_DATE`
- `dbt_valid_to = NULL::DATE`
- `is_current = TRUE`

**Hint:** Select from `students_new` and filter to only students whose old record was just closed (`dbt_valid_to = CURRENT_DATE` in snapshot_students).

In [ ]:
con.execute('''
    INSERT INTO snapshot_students
    SELECT
        new.student_id,
        new.name,
        new.email_address,
        new.phone_number,
        new.home_address,
        new.updated_at,
        CURRENT_DATE  AS dbt_valid_from,
        NULL::DATE    AS dbt_valid_to,
        TRUE          AS is_current
    FROM students_new new
    WHERE new.student_id IN (
        -- ✏️ Subquery: find student_ids whose old record was just closed today
        -- Hint: SELECT student_id FROM snapshot_students WHERE dbt_valid_to = CURRENT_DATE

    )
''')

print('New record versions inserted.')

### Step 6c: (Bonus) Insert brand-new students

In a real pipeline, the new file might also contain students who didn't exist before.
Insert any students from `students_new` who do not yet appear in `snapshot_students`.

In [ ]:
con.execute('''
    INSERT INTO snapshot_students
    SELECT
        student_id,
        name,
        email_address,
        phone_number,
        home_address,
        updated_at,
        CURRENT_DATE  AS dbt_valid_from,
        NULL::DATE    AS dbt_valid_to,
        TRUE          AS is_current
    FROM students_new
    WHERE student_id NOT IN (
        -- ✏️ Subquery: find all student_ids already in the snapshot

    )
''')

print('New students inserted (if any).')

---
## Step 7: Verify Your Snapshot

After all the updates, your snapshot should have **10 rows**:
- 6 original records (2 unchanged, still current; 4 closed)
- 4 new records (updated versions of the changed students)

Run the cells below to verify.

In [ ]:
# Full snapshot history, ordered by student and date
con.execute('''
    SELECT
        student_id,
        name,
        email_address,
        dbt_valid_from,
        dbt_valid_to,
        is_current
    FROM snapshot_students
    ORDER BY student_id, dbt_valid_from
''').df()

In [ ]:
# Current state only — should match students_new exactly (6 rows)
con.execute('''
    SELECT student_id, name, email_address, phone_number, home_address
    FROM snapshot_students
    WHERE is_current = TRUE
    ORDER BY student_id
''').df()

In [ ]:
# Students who have multiple versions (i.e. they changed)
con.execute('''
    SELECT
        student_id,
        name,
        email_address,
        dbt_valid_from,
        dbt_valid_to,
        is_current
    FROM snapshot_students
    WHERE student_id IN (
        SELECT student_id
        FROM snapshot_students
        GROUP BY student_id
        HAVING COUNT(*) > 1
    )
    ORDER BY student_id, dbt_valid_from
''').df()

In [ ]:
# Final row count check
total = con.execute('SELECT COUNT(*) FROM snapshot_students').fetchone()[0]
current = con.execute('SELECT COUNT(*) FROM snapshot_students WHERE is_current = TRUE').fetchone()[0]
historical = con.execute('SELECT COUNT(*) FROM snapshot_students WHERE is_current = FALSE').fetchone()[0]

print(f'Total rows:       {total}   (expected 10)')
print(f'Current records:  {current}  (expected 6)')
print(f'Historical rows:  {historical}  (expected 4)')

if total == 10 and current == 6 and historical == 4:
    print('\n✅ All checks passed!')
else:
    print('\n❌ Something looks off — check your UPDATE and INSERT statements.')

---
## Summary

You've implemented a **Slowly Changing Dimension Type 2** pipeline from scratch.

| Step | What you did |
|---|---|
| Load initial data | Created `students` and `dim_students` from a CSV |
| Create snapshot | Added SCD columns (`dbt_valid_from`, `dbt_valid_to`, `is_current`) |
| Detect changes | JOINed old vs new data to find differences |
| Close old records | UPDATEd changed rows with an end date and `is_current = FALSE` |
| Insert new versions | INSERTed updated rows with a new start date and `is_current = TRUE` |
| Handle new records | Inserted brand-new students who didn't exist before |

**In dbt, the `snapshot` block automates exactly these steps** — but now you understand what's happening under the hood.